In [8]:
# %%
# ------------------------------
# XGBoost Full Pipeline
# ------------------------------

import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor

# ------------------------------
# Load Dataset
# ------------------------------
df = pd.read_csv("corn_dataset_final.csv")

TRAIN_YEARS = [2016, 2017, 2018, 2019]
TEST_YEARS  = [2020, 2021, 2022]
TARGET_COL  = "YIELD"

train_df_all = df[df["year"].isin(TRAIN_YEARS)].copy()
test_df      = df[df["year"].isin(TEST_YEARS)].copy()

# ------------------------------
# Encode state
# ------------------------------
states = sorted(train_df_all["state_name"].unique())
state_to_idx = {s: i for i, s in enumerate(states)}

train_df_all["state_idx"] = train_df_all["state_name"].map(state_to_idx)
test_df["state_idx"]      = test_df["state_name"].map(state_to_idx)

# ------------------------------
# Feature columns
# ------------------------------
feature_cols = [
    "ppt (mm)_AVG", "tmax (degrees C)_AVG", "tmean (degrees C)_AVG", "tmin (degrees C)_AVG",
    "ssm_AVG", "susm_AVG",
    "EVI_AVG", "GCI_AVG", "NDWI_AVG", "NDVI_AVG",
    "AWC", "SOM", "CEC", "state_idx"
]

# ------------------------------
# Scaling
# ------------------------------
scaler_X = StandardScaler()
train_df_all[feature_cols] = scaler_X.fit_transform(train_df_all[feature_cols])
test_df[feature_cols]      = scaler_X.transform(test_df[feature_cols])

scaler_y = StandardScaler()
train_df_all["YIELD_STD"] = scaler_y.fit_transform(train_df_all[[TARGET_COL]])
test_df["YIELD_STD"]      = scaler_y.transform(test_df[[TARGET_COL]])

# ------------------------------
# Train / Validation Split
# ------------------------------
train_idx, val_idx = train_test_split(
    range(len(train_df_all)),
    test_size=0.2,
    random_state=42
)

X_train = train_df_all.iloc[train_idx][feature_cols].values
y_train = train_df_all.iloc[train_idx]["YIELD_STD"].values

X_val = train_df_all.iloc[val_idx][feature_cols].values
y_val = train_df_all.iloc[val_idx]["YIELD_STD"].values

# ------------------------------
# Initialize XGBoost
# ------------------------------
xgb_model = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

# ------------------------------
# Train
# ------------------------------
xgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=True
)

# ------------------------------
# Validation Evaluation
# ------------------------------
val_preds = xgb_model.predict(X_val)

val_mse = mean_squared_error(y_val, val_preds)
val_r2  = r2_score(y_val, val_preds)

val_preds_real = scaler_y.inverse_transform(val_preds.reshape(-1,1))
y_val_real     = scaler_y.inverse_transform(y_val.reshape(-1,1))

val_mse_real = mean_squared_error(y_val_real, val_preds_real)
val_r2_real  = r2_score(y_val_real, val_preds_real)

print("\n===== VALIDATION RESULTS =====")
print(f"MSE (scaled): {val_mse:.4f}")
print(f"R2  (scaled): {val_r2:.4f}")
print(f"MSE (real):   {val_mse_real:.4f}")
print(f"R2  (real):   {val_r2_real:.4f}")

# ------------------------------
# Save Model
# ------------------------------
joblib.dump(xgb_model, "best_xgboost_model.pkl")

# %%
# ------------------------------
# Test Evaluation (Year-wise)
# ------------------------------

xgb_model = joblib.load("best_xgboost_model.pkl")

YEARS = [2020]

for year in YEARS:
    year_df = test_df[test_df["year"] == year].reset_index(drop=True)

    if len(year_df) == 0:
        print(f"No data for year {year}")
        continue

    X_test = year_df[feature_cols].values
    y_test = year_df["YIELD_STD"].values

    preds_scaled = xgb_model.predict(X_test)

    preds_real = scaler_y.inverse_transform(preds_scaled.reshape(-1,1))
    y_real     = scaler_y.inverse_transform(y_test.reshape(-1,1))

    mse_scaled = mean_squared_error(y_test, preds_scaled)
    r2_scaled  = r2_score(y_test, preds_scaled)

    mse_real = mean_squared_error(y_real, preds_real)
    r2_real  = r2_score(y_real, preds_real)

    print(f"\n--- Year {year} ---")
    print(f"MSE (scaled): {mse_scaled:.4f}, R2 (scaled): {r2_scaled:.4f}")
    print(f"MSE (real):   {mse_real:.4f}, R2 (real):   {r2_real:.4f}")


[0]	validation_0-rmse:0.99268
[1]	validation_0-rmse:0.98731
[2]	validation_0-rmse:0.98248
[3]	validation_0-rmse:0.97751
[4]	validation_0-rmse:0.97269
[5]	validation_0-rmse:0.96820
[6]	validation_0-rmse:0.96336
[7]	validation_0-rmse:0.95899
[8]	validation_0-rmse:0.95396
[9]	validation_0-rmse:0.94946
[10]	validation_0-rmse:0.94551
[11]	validation_0-rmse:0.94084
[12]	validation_0-rmse:0.93707
[13]	validation_0-rmse:0.93245
[14]	validation_0-rmse:0.92857
[15]	validation_0-rmse:0.92448
[16]	validation_0-rmse:0.92022
[17]	validation_0-rmse:0.91611
[18]	validation_0-rmse:0.91186
[19]	validation_0-rmse:0.90771
[20]	validation_0-rmse:0.90363
[21]	validation_0-rmse:0.89961
[22]	validation_0-rmse:0.89548
[23]	validation_0-rmse:0.89163
[24]	validation_0-rmse:0.88781
[25]	validation_0-rmse:0.88384
[26]	validation_0-rmse:0.88053
[27]	validation_0-rmse:0.87752
[28]	validation_0-rmse:0.87375
[29]	validation_0-rmse:0.87038
[30]	validation_0-rmse:0.86674
[31]	validation_0-rmse:0.86341
[32]	validation_0-